# M5 Forecasting — Notebook 3: LightGBM Quantile Forecasting

**Pipeline:** EDA → Feature Engineering → **Forecasting** → Optimization

Trains q10 / q50 / q90 models with walk-forward CV, SHAP, and MLflow tracking.
Saves `forecasts.parquet` for the optimization notebook.

**Prerequisite:** Run `02_feature_engineering.ipynb` first.


In [ ]:
import sys, pathlib

SRC_ROOT = '/kaggle/input/youssefmousaaid/m5-forecast-optimize-src-files/m5-forecast-optimize'
DATA_DIR = '/kaggle/input/competitions/m5-forecasting-accuracy/'

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

print(f'src/ present : {(pathlib.Path(SRC_ROOT) / "src").exists()}')
print(f'DATA_DIR     : {DATA_DIR}')


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import mlflow
import shap
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')

from src.data.features import FEATURE_COLS, TARGET_COL, DATE_COL
from src.forecasting.lgbm_quantile import (
    walk_forward_splits, wrmsse, dollar_error_cost,
    train_quantile_models, predict_quantiles,
    QUANTILES, HORIZON, BASE_PARAMS,
)
from src.optimization.newsvendor import optimal_order_quantity, CostParams

COLORS = {'green':'#48bb78','blue':'#63b3ed','orange':'#f6ad55',
          'red':'#fc8181','gray':'#718096','purple':'#b794f4'}
plt.rcParams.update({
    'figure.facecolor':'#0f1117','axes.facecolor':'#1a202c',
    'axes.edgecolor':'#2d3748','axes.labelcolor':'#e2e8f0',
    'xtick.color':'#a0aec0','ytick.color':'#a0aec0',
    'text.color':'#e2e8f0','grid.color':'#2d3748',
})
N_CV_SPLITS = 3
print(f'Quantiles : {QUANTILES}')
print(f'Horizon   : {HORIZON} days')
print('Imports OK ✓')


## 1. Load Feature Matrix

In [ ]:
df = pd.read_parquet('/kaggle/working/features.parquet')
df['date'] = pd.to_datetime(df['date'])
print(f'Rows     : {len(df):,}')
print(f'Items    : {df["id"].nunique():,}')
print(f'Features : {len(FEATURE_COLS)}')
print(f'Range    : {df["date"].min().date()} -> {df["date"].max().date()}')


## 2. Walk-Forward CV Splits

In [ ]:
splits = list(walk_forward_splits(df, n_splits=N_CV_SPLITS))
print(f'{"Fold":<6} {"Train rows":>12} {"Val rows":>10} {"Train end":>12} {"Val start":>12} {"Val end":>12}')
print('-' * 68)
for i, (tr_idx, val_idx) in enumerate(splits):
    tr  = df.loc[tr_idx,  'date']
    val = df.loc[val_idx, 'date']
    print(f'{i:<6} {len(tr_idx):>12,} {len(val_idx):>10,} '
          f'{str(tr.max().date()):>12} '
          f'{str(val.min().date()):>12} '
          f'{str(val.max().date()):>12}')


## 3. Train Quantile Models

In [ ]:
os.makedirs('/kaggle/working/models', exist_ok=True)
mlflow.set_tracking_uri('/kaggle/working/mlruns')

models = train_quantile_models(
    df,
    output_dir      = '/kaggle/working/models',
    experiment_name = 'm5_lgbm_quantile',
    n_cv_splits     = N_CV_SPLITS,
)
print('Models trained:', list(models.keys()))


## 4. CV Metrics

In [ ]:
results = []
for q, model in models.items():
    for fold, (tr_idx, val_idx) in enumerate(walk_forward_splits(df, n_splits=N_CV_SPLITS)):
        X_val = df.loc[val_idx, FEATURE_COLS]
        y_val = df.loc[val_idx, TARGET_COL].values
        preds = np.maximum(model.predict(X_val), 0)
        dcost = dollar_error_cost(y_val, preds)
        results.append({'quantile':q, 'fold':fold,
                        'WRMSSE':wrmsse(y_val, preds),
                        'dollar_cost':dcost['total_cost_usd']})

results_df = pd.DataFrame(results)
print(results_df.groupby('quantile')[['WRMSSE','dollar_cost']].mean().round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for q, color in zip(QUANTILES, [COLORS['blue'],COLORS['green'],COLORS['orange']]):
    qdf = results_df[results_df['quantile']==q]
    axes[0].plot(qdf['fold'], qdf['WRMSSE'], marker='o', color=color, label=f'q={q}')
    axes[1].plot(qdf['fold'], qdf['dollar_cost'], marker='o', color=color, label=f'q={q}')
axes[0].set_title('WRMSSE per Fold'); axes[0].set_xlabel('Fold'); axes[0].legend()
axes[1].set_title('Dollar Error Cost per Fold'); axes[1].set_xlabel('Fold'); axes[1].legend()
plt.tight_layout(); plt.show()


## 5. Generate 28-Day Forecasts

In [ ]:
last_date = df['date'].max()
cutoff    = last_date - pd.Timedelta(days=HORIZON - 1)
test_df   = df[df['date'] >= cutoff].copy().reset_index(drop=True)

preds_df    = predict_quantiles(models, test_df)
forecast_df = pd.concat([
    test_df[['id','store_id','item_id','date','sales']].reset_index(drop=True),
    preds_df.reset_index(drop=True)
], axis=1)

print(f'Forecast rows: {len(forecast_df):,}')
print(f'Horizon      : {forecast_df["date"].min().date()} -> {forecast_df["date"].max().date()}')
forecast_df.head()


In [ ]:
sample_items = forecast_df['id'].unique()[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('28-Day Probabilistic Forecasts (blue band = 80% PI)', fontsize=13)

for ax, iid in zip(axes.flatten(), sample_items):
    idf = forecast_df[forecast_df['id']==iid].sort_values('date')
    ax.fill_between(idf['date'], idf['q10'], idf['q90'],
                    alpha=0.25, color=COLORS['blue'])
    ax.plot(idf['date'], idf['q50'], color=COLORS['blue'], lw=2, label='Median')
    ax.scatter(idf['date'], idf['sales'], color=COLORS['green'], s=15, marker='x', label='Actual')
    ax.set_title(f'{idf["store_id"].iloc[0]}', fontsize=9)
    ax.tick_params(axis='x', rotation=30, labelsize=7)
axes[0][0].legend(fontsize=7)
plt.tight_layout(); plt.show()


## 6. SHAP Feature Importance

In [ ]:
sample_shap = df.sample(min(3000, len(df)), random_state=42)
explainer   = shap.TreeExplainer(models[0.50])
shap_values = explainer.shap_values(sample_shap[FEATURE_COLS])

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, sample_shap[FEATURE_COLS], max_display=20, show=False)
plt.title('SHAP Beeswarm — q50 model', fontsize=12)
plt.tight_layout()
plt.savefig('/kaggle/working/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0),
                           index=FEATURE_COLS).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 10))
bar_colors = [COLORS['green'] if 'lag' in f or 'rmean' in f
              else COLORS['blue'] if 'price' in f
              else COLORS['orange'] for f in mean_abs_shap.index]
ax.barh(mean_abs_shap.index, mean_abs_shap.values, color=bar_colors, edgecolor='none')
ax.set_title('Mean |SHAP| — green=lag/rolling  blue=price  orange=calendar', fontsize=10)
ax.set_xlabel('Mean |SHAP value|')
plt.tight_layout(); plt.show()


## 7. Save Forecasts

In [ ]:
cost_p = CostParams()
forecast_df['q_star'] = optimal_order_quantity(
    forecast_df['q10'].values,
    forecast_df['q50'].values,
    forecast_df['q90'].values,
    cost=cost_p,
)

import numpy as np
rng = np.random.default_rng(42)
forecast_df['inventory'] = np.round(
    forecast_df['q90'].values * rng.uniform(0.8, 1.4, len(forecast_df))
).astype(int)

forecast_df.to_parquet('/kaggle/working/forecasts.parquet', index=False)
print(f'Saved -> /kaggle/working/forecasts.parquet')
print(f'Columns: {list(forecast_df.columns)}')
print()
print('Next -> 04_optimization.ipynb')
